# Instacart PyHercules Clustering Analysis

This notebook replicates the PyHercules clustering process for the Instacart dataset:
1. Load and preprocess data (one-hot encoding, standard scaling)
2. Standard K-means clustering (k=7, single level)
3. Feature ranking by positive z-score/deviation only (top 10 positive characteristics)
4. LLM-generated cluster descriptions using Google Gemini
5. Export cluster details and feature rankings to CSV

In [76]:
# Import required libraries
import pandas as pd
import numpy as np
import os
import json
import re
import warnings
from typing import Dict, List, Any, Tuple, Optional
from datetime import datetime

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

import google.generativeai as genai

warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


## 1. Load Data

In [77]:
# Load the Instacart dataset
data_path = r'C:\Users\weezh\OneDrive\Desktop\pyhercules\dataset\8th exploration instacart\instacart_user_features.csv'

df = pd.read_csv(data_path)
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

# Drop time-preference/consistency columns before clustering
drop_columns = [
    'preferred_day_of_week',
    'preferred_hour_of_day',
    'day_consistency',
    'hour_consistency',
    'frequency_consistency',
]
existing_drop_columns = [col for col in drop_columns if col in df.columns]
df = df.drop(columns=existing_drop_columns)
print(f"Dropped columns: {existing_drop_columns}")
print(f"Remaining columns after drop: {df.shape[1]}")

print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Dataset loaded: 206209 rows, 37 columns
Dropped columns: ['preferred_day_of_week', 'preferred_hour_of_day', 'day_consistency', 'hour_consistency', 'frequency_consistency']
Remaining columns after drop: 32

Column names:
['user_id', 'total_orders', 'avg_days_between_orders', 'avg_basket_size', 'reorder_rate', 'total_products_purchased', 'unique_products', 'unique_aisles', 'unique_departments', 'product_exploration_rate', 'avg_products_per_aisle', 'dept_pct_alcohol', 'dept_pct_babies', 'dept_pct_bakery', 'dept_pct_beverages', 'dept_pct_breakfast', 'dept_pct_bulk', 'dept_pct_canned_goods', 'dept_pct_dairy_eggs', 'dept_pct_deli', 'dept_pct_dry_goods_pasta', 'dept_pct_frozen', 'dept_pct_household', 'dept_pct_international', 'dept_pct_meat_seafood', 'dept_pct_missing', 'dept_pct_other', 'dept_pct_pantry', 'dept_pct_personal_care', 'dept_pct_pets', 'dept_pct_produce', 'dept_pct_snacks']

First few rows:


,user_id,total_orders,avg_days_between_orders,avg_basket_size,reorder_rate,total_products_purchased,unique_products,unique_aisles,unique_departments,product_exploration_rate,...,dept_pct_household,dept_pct_international,dept_pct_meat_seafood,dept_pct_missing,dept_pct_other,dept_pct_pantry,dept_pct_personal_care,dept_pct_pets,dept_pct_produce,dept_pct_snacks
0,1,10,19.555556,5.900000,0.694915,59,18,12,7,0.305085,...,0.033898,0.000000,0.000000,0.0,0.0,0.016949,0.000000,0.0,0.084746,0.372881
1,2,14,15.230769,13.928571,0.476923,195,102,33,13,0.523077,...,0.000000,0.015385,0.005128,0.0,0.0,0.056410,0.005128,0.0,0.184615,0.215385
2,3,12,12.090909,7.333333,0.625000,88,33,16,9,0.375000,...,0.011364,0.000000,0.000000,0.0,0.0,0.045455,0.000000,0.0,0.431818,0.102273
3,4,5,13.750000,3.600000,0.055556,18,17,14,9,0.944444,...,0.111111,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.111111,0.055556
4,5,4,13.333333,9.250000,0.378378,37,23,16,9,0.621622,...,0.000000,0.054054,0.000000,0.0,0.0,0.054054,0.000000,0.0,0.513514,0.027027


## 2. Data Preprocessing (Copy from PyHercules)

In [78]:
def detect_column_types(df: pd.DataFrame) -> Dict[str, str]:
    """
    Automatically detects column types as 'numeric', 'categorical', or 'text'.
    """
    types = {}
    
    for col in df.columns:
        # 1. Try to detect if column is actually numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            types[col] = 'numeric'
            continue
        
        # Try converting to numeric
        try:
            numeric_converted = pd.to_numeric(df[col], errors='coerce')
            non_null_ratio = numeric_converted.notna().sum() / len(df[col])
            
            if non_null_ratio > 0.8:
                types[col] = 'numeric'
                continue
        except:
            pass
        
        # 2. For object/string columns, determine if categorical or text
        if df[col].dtype == 'object' or df[col].dtype.name == 'category':
            n_unique = df[col].nunique()
            n_total = len(df[col].dropna())
            
            if n_total == 0:
                types[col] = 'text'
                continue
                
            cardinality_ratio = n_unique / n_total
            avg_length = df[col].astype(str).str.len().mean()
            
            is_low_cardinality = cardinality_ratio < 0.05 or n_unique < 50
            is_short = avg_length < 30
            
            if is_low_cardinality and is_short:
                types[col] = 'categorical'
            else:
                types[col] = 'text'
        else:
            types[col] = 'text'
    
    return types


def preprocess_mixed_data(df: pd.DataFrame, 
                          column_types: Dict[str, str],
                          variable_metadata: Optional[Dict[str, Dict[str, Any]]] = None
                          ) -> Tuple[np.ndarray, List[str], Dict[str, Dict[str, Any]]]:
    """
    Preprocesses mixed data types (numeric, categorical, text) into a unified numerical representation.
    """
    processed_parts = []
    final_column_names = []
    metadata_out = variable_metadata.copy() if variable_metadata else {}
    
    for col, col_type in column_types.items():
        if col not in df.columns:
            continue
            
        if col_type == 'numeric':
            # Keep numeric columns as-is
            col_data = pd.to_numeric(df[col], errors='coerce')
            col_mean = col_data.mean()
            col_data = col_data.fillna(col_mean if not pd.isna(col_mean) else 0)
            processed_parts.append(col_data.values.reshape(-1, 1))
            final_column_names.append(col)
            if col not in metadata_out:
                metadata_out[col] = {'name': col, 'type': 'numeric', 'source_column': col}
                
        elif col_type == 'categorical':
            # One-hot encode categorical columns
            col_data = df[col].fillna('_MISSING_')
            encoded = pd.get_dummies(col_data, prefix=col, drop_first=False, dtype=float)
            if encoded.shape[1] > 0:
                processed_parts.append(encoded.values)
                for new_col in encoded.columns:
                    final_column_names.append(new_col)
                    category_value = new_col.replace(f"{col}_", "")
                    metadata_out[new_col] = {
                        'name': new_col,
                        'type': 'categorical_encoded',
                        'source_column': col,
                        'category': category_value
                    }
        
        elif col_type == 'text':
            # TF-IDF + SVD for text columns
            texts = df[col].fillna('').astype(str)
            
            if texts.str.strip().eq('').all():
                continue
            
            try:
                vectorizer = TfidfVectorizer(
                    max_features=100,
                    stop_words='english',
                    min_df=1,
                    max_df=0.95
                )
                tfidf_matrix = vectorizer.fit_transform(texts)
                
                n_components = min(10, tfidf_matrix.shape[1] - 1, tfidf_matrix.shape[0] - 1)
                if n_components > 0:
                    svd = TruncatedSVD(n_components=n_components, random_state=42)
                    text_features = svd.fit_transform(tfidf_matrix)
                    
                    processed_parts.append(text_features)
                    for i in range(text_features.shape[1]):
                        new_col_name = f"{col}_text_dim{i}"
                        final_column_names.append(new_col_name)
                        metadata_out[new_col_name] = {
                            'name': new_col_name,
                            'type': 'text_embedding',
                            'source_column': col,
                            'method': 'tfidf_svd',
                            'dimension': i
                        }
            except Exception as e:
                warnings.warn(f"Error processing text column '{col}': {e}. Skipping.")
                continue
    
    if not processed_parts:
        raise ValueError("No valid columns to process after preprocessing.")
    
    # Combine all processed parts
    combined_data = np.hstack(processed_parts)
    
    return combined_data, final_column_names, metadata_out


# Detect column types
column_types = detect_column_types(df)
print(f"\nDetected column types:")
for col, ctype in column_types.items():
    print(f"  {col}: {ctype}")

# Preprocess data
preprocessed_data, feature_names, metadata = preprocess_mixed_data(df, column_types)
print(f"\nPreprocessed data shape: {preprocessed_data.shape}")
print(f"Number of features after preprocessing: {len(feature_names)}")


Detected column types:
  user_id: numeric
  total_orders: numeric
  avg_days_between_orders: numeric
  avg_basket_size: numeric
  reorder_rate: numeric
  total_products_purchased: numeric
  unique_products: numeric
  unique_aisles: numeric
  unique_departments: numeric
  product_exploration_rate: numeric
  avg_products_per_aisle: numeric
  dept_pct_alcohol: numeric
  dept_pct_babies: numeric
  dept_pct_bakery: numeric
  dept_pct_beverages: numeric
  dept_pct_breakfast: numeric
  dept_pct_bulk: numeric
  dept_pct_canned_goods: numeric
  dept_pct_dairy_eggs: numeric
  dept_pct_deli: numeric
  dept_pct_dry_goods_pasta: numeric
  dept_pct_frozen: numeric
  dept_pct_household: numeric
  dept_pct_international: numeric
  dept_pct_meat_seafood: numeric
  dept_pct_missing: numeric
  dept_pct_other: numeric
  dept_pct_pantry: numeric
  dept_pct_personal_care: numeric
  dept_pct_pets: numeric
  dept_pct_produce: numeric
  dept_pct_snacks: numeric

Preprocessed data shape: (206209, 32)
Number of

In [79]:
# Apply standard scaling (like Hercules does internally)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(preprocessed_data)

print(f"Data scaled. Shape: {scaled_data.shape}")
print(f"Sample statistics after scaling:")
print(f"  Mean: {scaled_data.mean():.6f}")
print(f"  Std: {scaled_data.std():.6f}")

Data scaled. Shape: (206209, 32)
Sample statistics after scaling:
  Mean: -0.000000
  Std: 1.000000


## 3. K-Means Clustering

In [80]:
# Single-level clustering: k=7
k_clusters = 7

print(f"Starting clustering with k={k_clusters}...")
kmeans = KMeans(n_clusters=k_clusters, random_state=42, n_init=10, max_iter=300)
cluster_labels = kmeans.fit_predict(scaled_data)

print("Clustering complete!")
print(f"  Number of clusters: {k_clusters}")
print(f"  Cluster sizes: min={np.bincount(cluster_labels).min()}, max={np.bincount(cluster_labels).max()}, mean={np.bincount(cluster_labels).mean():.1f}")

Starting clustering with k=7...
Clustering complete!
  Number of clusters: 7
  Cluster sizes: min=7518, max=54860, mean=29458.4


In [81]:
# Display cluster distribution
print("\nCluster distribution:")
for i in range(k_clusters):
    cluster_size = np.sum(cluster_labels == i)
    percentage = (cluster_size / len(cluster_labels)) * 100
    print(f"  Cluster {i}: {cluster_size} data points ({percentage:.1f}%)")


Cluster distribution:
  Cluster 0: 45878 data points (22.2%)
  Cluster 1: 7518 data points (3.6%)
  Cluster 2: 54860 data points (26.6%)
  Cluster 3: 20211 data points (9.8%)
  Cluster 4: 15555 data points (7.5%)
  Cluster 5: 29607 data points (14.4%)
  Cluster 6: 32580 data points (15.8%)


## 4. Compute Statistics and Rank by Positive Z-Score/Deviation

In [82]:
def compute_cluster_statistics(data: np.ndarray, 
                               labels: np.ndarray, 
                               feature_names: List[str],
                               global_means: np.ndarray,
                               level: int) -> pd.DataFrame:
    """
    Compute statistics for each cluster and rank features by deviation.
    Skips identifier columns like user_id, id, etc.
    """
    unique_labels = np.unique(labels)
    all_stats = []
    
    for cluster_id in unique_labels:
        cluster_mask = labels == cluster_id
        cluster_data = data[cluster_mask]
        
        for i, feature_name in enumerate(feature_names):
            # Skip identifier columns (user_id, id, etc.)
            feature_lower = feature_name.lower()
            if any(pattern in feature_lower for pattern in ['user_id', 'userid', '_id', 'id_', 'customer_id', 'customerid']):
                continue
            
            feature_data = cluster_data[:, i]
            feature_data_clean = feature_data[~np.isnan(feature_data)]
            
            if feature_data_clean.size == 0:
                continue
            
            cluster_mean = float(np.mean(feature_data_clean))
            global_mean = float(global_means[i])
            
            # Calculate percentage deviation
            if abs(global_mean) > 1e-10:
                deviation_pct = ((cluster_mean - global_mean) / abs(global_mean)) * 100
            else:
                deviation_pct = cluster_mean * 100
            
            abs_deviation_pct = abs(deviation_pct)
            
            all_stats.append({
                'level': level,
                'cluster_id': int(cluster_id),
                'feature_name': feature_name,
                'cluster_mean': cluster_mean,
                'global_mean': global_mean,
                'deviation_pct': deviation_pct,
                'abs_deviation_pct': abs_deviation_pct,
                'cluster_size': int(cluster_mask.sum())
            })
    
    stats_df = pd.DataFrame(all_stats)
    return stats_df


def get_top_features_per_cluster(stats_df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    """
    Remove negative deviations, then get top N features by non-negative deviation for each cluster.
    """
    non_negative_features = stats_df[stats_df['deviation_pct'] >= 0].copy()
    if non_negative_features.empty:
        return non_negative_features
    top_features = non_negative_features.sort_values('deviation_pct', ascending=False).groupby('cluster_id').head(top_n)
    return top_features


# Compute global means from unscaled data
global_means = np.mean(preprocessed_data, axis=0)

print("Computing statistics for clusters (excluding ID columns)...")
cluster_stats = compute_cluster_statistics(
    preprocessed_data, cluster_labels, feature_names, global_means, level=1
)

# Remove negatives, then get top 10 features for each cluster
top_features = get_top_features_per_cluster(cluster_stats, top_n=10)

print(f"\nCluster statistics: {len(cluster_stats)} feature-cluster pairs (ID columns excluded)")
print(f"Top 10 non-negative features: {len(top_features)} (up to 10 per cluster after removing negatives)")

Computing statistics for clusters (excluding ID columns)...

Cluster statistics: 217 feature-cluster pairs (ID columns excluded)
Top 10 non-negative features: 62 (up to 10 per cluster after removing negatives)


In [83]:
# Display sample top features
print("\nSample top features for Cluster 0:")
top_features[top_features['cluster_id'] == 0].head()


Sample top features for Cluster 0:


,level,cluster_id,feature_name,cluster_mean,global_mean,deviation_pct,abs_deviation_pct,cluster_size
22,1,0,dept_pct_international,0.015931,0.008016,98.725075,98.725075,45878
16,1,0,dept_pct_canned_goods,0.059075,0.032230,83.290784,83.290784,45878
19,1,0,dept_pct_dry_goods_pasta,0.043564,0.025157,73.169843,73.169843,45878
26,1,0,dept_pct_pantry,0.095762,0.060683,57.805534,57.805534,45878
23,1,0,dept_pct_meat_seafood,0.031524,0.022273,41.537406,41.537406,45878


## 5. Configure Google Gemini API

In [ ]:
# Configure Google Gemini API
GOOGLE_API_KEY = ""

genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-2.5-flash-lite')

print("Google Gemini API configured successfully!")

Google Gemini API configured successfully!


## 6. Generate LLM Descriptions for Clusters

In [85]:
def build_cluster_prompt(cluster_id: int, 
                        top_features_df: pd.DataFrame,
                        level: int,
                        sample_data: np.ndarray,
                        sample_size: int = 5) -> str:
    """
    Build LLM prompt for a single cluster using top non-negative features.
    """
    cluster_features = top_features_df[top_features_df['cluster_id'] == cluster_id].sort_values(
        'deviation_pct', ascending=False
    ).head(10)
    
    cluster_size = cluster_features.iloc[0]['cluster_size'] if len(cluster_features) > 0 else 0
    
    prompt = f"""Generate a concise 'title' (max 5-7 words) and 'description' (3-4 sentences) for the Cluster below (Level {level}).

Industry Context: You are analyzing customer purchase behavior in the online grocery delivery market.
Business Goal: Identify distinct customer segments to power hyper-personalized recommendations and targeted email campaigns.

RESPONSE FORMAT: Respond ONLY with a single, valid JSON object.
- Top-level key MUST be the string representation of the 'Cluster ID' provided (e.g., \"{cluster_id}\").
- Value MUST be a JSON object containing non-empty \"title\" and \"description\" string keys.

EXAMPLE:
{{
  \"{cluster_id}\": {{ \"title\": \"Example Title\", \"description\": \"Example description.\" }}
}}

IMPORTANT: Ensure the entire output is valid JSON. Do NOT include markdown fences (```json ... ```) or any text outside the JSON structure.

--- Cluster Information ---

--- Cluster ID: {cluster_id} (L{level}, {cluster_size} base items) ---

Most Defining Features (Top Non-Negative Deviations from Global Mean):
"""
    
    if cluster_features.empty:
        prompt += "- No non-negative deviations identified among selected features.\n"
    else:
        for idx, row in cluster_features.iterrows():
            prompt += f"- {row['feature_name']}: (Cluster: {row['cluster_mean']:.2f} vs Global: {row['global_mean']:.2f}) | Deviation: {row['deviation_pct']:.2f}%\n"
    
    prompt += f"\n--- End Cluster ID: {cluster_id} ---\n"
    prompt += f"\n--- End Clusters Information ---\n"
    prompt += f"\nGenerate the JSON output for the Cluster ID: {cluster_id}"
    
    return prompt


def parse_llm_response(llm_output: str, cluster_id: int) -> Tuple[Optional[str], Optional[str]]:
    """
    Parse LLM response to extract title and description.
    """
    if not llm_output:
        return None, None
    
    # Try to extract JSON from markdown fences
    json_match = re.search(r'```(?:json)?\s*\n?({.*?})\s*\n?```', llm_output, re.DOTALL)
    if json_match:
        llm_output = json_match.group(1)
    
    # Try to parse JSON
    try:
        parsed = json.loads(llm_output)
        cluster_key = str(cluster_id)
        
        if cluster_key in parsed and isinstance(parsed[cluster_key], dict):
            title = parsed[cluster_key].get('title', '')
            description = parsed[cluster_key].get('description', '')
            return title, description
    except json.JSONDecodeError:
        pass
    
    return None, None


def generate_descriptions_for_level(cluster_ids: List[int],
                                   top_features_df: pd.DataFrame,
                                   level: int,
                                   data: np.ndarray,
                                   max_retries: int = 3,
                                   save_prompts: bool = True,
                                   prompt_log_dir: Optional[str] = None) -> pd.DataFrame:
    """
    Generate LLM descriptions for all clusters at a given level.
    Retries on parsing failures up to max_retries times.
    Optionally saves every prompt sent to the LLM.
    """
    results = []
    prompt_index = []
    run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    if save_prompts:
        if prompt_log_dir is None:
            prompt_log_dir = r'C:\Users\weezh\OneDrive\Desktop\pyhercules\evaluation\LLM Silhouette Score Non Negative\prompt_logs'
        os.makedirs(prompt_log_dir, exist_ok=True)
        print(f"Saving prompts to: {prompt_log_dir}")
    
    for i, cluster_id in enumerate(cluster_ids):
        print(f"Processing Level {level}, Cluster {cluster_id} ({i+1}/{len(cluster_ids)})...")
        
        # Build prompt
        prompt = build_cluster_prompt(cluster_id, top_features_df, level, data)
        
        if save_prompts:
            prompt_filename = f"instacart_non_negative_prompt_level{level}_cluster{cluster_id}_{run_timestamp}.txt"
            prompt_path = os.path.join(prompt_log_dir, prompt_filename)
            with open(prompt_path, 'w', encoding='utf-8') as f:
                f.write(prompt)
            prompt_index.append({
                'level': level,
                'cluster_id': cluster_id,
                'prompt_file': prompt_path
            })
        
        # Retry logic
        title, description = None, None
        for attempt in range(max_retries):
            try:
                response = model.generate_content(prompt)
                llm_output = response.text
                
                # Parse response
                title, description = parse_llm_response(llm_output, cluster_id)
                
                if title and description:
                    results.append({
                        'level': level,
                        'cluster_id': cluster_id,
                        'title': title,
                        'description': description
                    })
                    print(f"  ✓ Success: {title[:50]}...")
                    break  # Success, exit retry loop
                else:
                    if attempt < max_retries - 1:
                        print(f"  ⟳ Retry {attempt + 1}/{max_retries - 1}: Failed to parse, retrying...")
                    else:
                        print(f"  ✗ Failed to parse response after {max_retries} attempts")
                        results.append({
                            'level': level,
                            'cluster_id': cluster_id,
                            'title': f"Cluster {cluster_id} (Level {level})",
                            'description': "Description generation failed after retries."
                        })
            except Exception as e:
                if attempt < max_retries - 1:
                    print(f"  ⟳ Retry {attempt + 1}/{max_retries - 1}: Error ({e}), retrying...")
                else:
                    print(f"  ✗ Error after {max_retries} attempts: {e}")
                    results.append({
                        'level': level,
                        'cluster_id': cluster_id,
                        'title': f"Cluster {cluster_id} (Level {level})",
                        'description': f"Error: {str(e)}"
                    })
    
    if save_prompts and prompt_index:
        index_path = os.path.join(
            prompt_log_dir,
            f"instacart_non_negative_prompt_index_level{level}_{run_timestamp}.json"
        )
        with open(index_path, 'w', encoding='utf-8') as f:
            json.dump(prompt_index, f, indent=2)
        print(f"Prompt index saved to: {index_path}")
    
    return pd.DataFrame(results)


print("Generating descriptions for clusters...")
cluster_descriptions = generate_descriptions_for_level(
    cluster_ids=list(range(k_clusters)),
    top_features_df=top_features,
    level=1,
    data=preprocessed_data,
    save_prompts=True
)

print(f"\n{'='*80}")
print(f"Complete: {len(cluster_descriptions)} clusters processed")
print(f"{'='*80}\n")

Generating descriptions for clusters...
Saving prompts to: C:\Users\weezh\OneDrive\Desktop\pyhercules\evaluation\LLM Silhouette Score Non Negative\prompt_logs
Processing Level 1, Cluster 0 (1/7)...
  ✓ Success: Pantry Staples & Explorer Shoppers...
Processing Level 1, Cluster 1 (2/7)...
  ✓ Success: Household & Personal Care Enthusiasts...
Processing Level 1, Cluster 2 (3/7)...
  ✓ Success: Engaged, Diverse Basket Shoppers...
Processing Level 1, Cluster 3 (4/7)...
  ✓ Success: High-Volume, Diverse Product Buyers...
Processing Level 1, Cluster 4 (5/7)...
  ✓ Success: Beverage & Snack Enthusiasts...
Processing Level 1, Cluster 5 (6/7)...
  ✓ Success: Fresh Produce Lovers, Loyal Buyers...
Processing Level 1, Cluster 6 (7/7)...
  ✓ Success: Fresh & Frozen Meal Enthusiasts...
Prompt index saved to: C:\Users\weezh\OneDrive\Desktop\pyhercules\evaluation\LLM Silhouette Score Non Negative\prompt_logs\instacart_non_negative_prompt_index_level1_20260314_022031.json

Complete: 7 clusters processed

In [86]:
# All clustering complete - ready for export
print("All descriptions generated successfully!")

All descriptions generated successfully!


## 7. Export Results to CSV

In [87]:
# Prepare descriptions for export
all_descriptions = cluster_descriptions.sort_values(['level', 'cluster_id'])

print(f"Total clusters with descriptions: {len(all_descriptions)}")
print(f"\nSample descriptions:")
all_descriptions.head(10)

Total clusters with descriptions: 7

Sample descriptions:


,level,cluster_id,title,description
0,1,0,Pantry Staples & Explorer Shoppers,"This segment actively explores new products, s..."
1,1,1,Household & Personal Care Enthusiasts,This segment heavily prioritizes household ess...
2,1,2,"Engaged, Diverse Basket Shoppers",These customers exhibit a high engagement with...
3,1,3,"High-Volume, Diverse Product Buyers","These customers are exceptionally active, plac..."
4,1,4,Beverage & Snack Enthusiasts,This segment heavily favors alcoholic beverage...
5,1,5,"Fresh Produce Lovers, Loyal Buyers",This segment shows a very strong preference fo...
6,1,6,Fresh & Frozen Meal Enthusiasts,This segment heavily favors fresh items like d...


In [88]:
# Save cluster details CSV
output_dir = r'C:\Users\weezh\OneDrive\Desktop\pyhercules\evaluation\LLM Silhouette Score Non Negative'
os.makedirs(output_dir, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

cluster_details_path = os.path.join(output_dir, f'instacart_non_negative_cluster_details_{timestamp}.csv')
all_descriptions.to_csv(cluster_details_path, index=False)
print(f"\n✓ Cluster details saved to: {cluster_details_path}")


✓ Cluster details saved to: C:\Users\weezh\OneDrive\Desktop\pyhercules\evaluation\LLM Silhouette Score Non Negative\instacart_non_negative_cluster_details_20260314_022040.csv


In [89]:
# Save top features (positive defining characteristics) CSV
all_top_features = top_features.sort_values(['level', 'cluster_id', 'deviation_pct'], ascending=[True, True, False])

top_features_path = os.path.join(output_dir, f'instacart_non_negative_top_features_{timestamp}.csv')
all_top_features.to_csv(top_features_path, index=False)
print(f"✓ Top positive defining features saved to: {top_features_path}")

✓ Top positive defining features saved to: C:\Users\weezh\OneDrive\Desktop\pyhercules\evaluation\LLM Silhouette Score Non Negative\instacart_non_negative_top_features_20260314_022040.csv


In [90]:
# Display summary statistics
print(f"\n{'='*80}")
print(f"SUMMARY")
print(f"{'='*80}")
print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} original columns")
print(f"Preprocessed features: {len(feature_names)}")
print(f"\nClustering:")
print(f"  k={k_clusters} clusters (single level)")
print(f"\nLLM Descriptions:")
print(f"  Total clusters described: {len(all_descriptions)}")
print(f"  Successful: {len(all_descriptions[all_descriptions['description'] != 'Description generation failed.'])}")
print(f"\nOutput files:")
print(f"  1. {cluster_details_path}")
print(f"  2. {top_features_path}")
print(f"{'='*80}")


SUMMARY
Dataset: 206209 rows, 32 original columns
Preprocessed features: 32

Clustering:
  k=7 clusters (single level)

LLM Descriptions:
  Total clusters described: 7
  Successful: 7

Output files:
  1. C:\Users\weezh\OneDrive\Desktop\pyhercules\evaluation\LLM Silhouette Score Non Negative\instacart_non_negative_cluster_details_20260314_022040.csv
  2. C:\Users\weezh\OneDrive\Desktop\pyhercules\evaluation\LLM Silhouette Score Non Negative\instacart_non_negative_top_features_20260314_022040.csv
